In [ ]:
# --- Célula 1: Coleta e Diagnóstico Inicial dos Dados Brutos ---

!pip install -q yfinance pandas_datareader

import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import pandas_datareader.data as web
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO DAS DATAS (MODIFICADO PARA SER DINÂMICO) ---
# Define a data final como o dia de hoje.
# A biblioteca yfinance buscará o último dado disponível até esta data.
end_date = datetime.today().strftime('%Y-%m-%d')

# O start_date continua sendo 5 anos antes para manter a janela de features.
start_date = (pd.to_datetime(end_date) - pd.DateOffset(years=5)).strftime('%Y-%m-%d')
print(f"A recolher dados de {start_date} até {end_date}...")


# --- ETAPA 1: BAIXAR DADOS DE MERCADO DO YAHOO FINANCE ---
tickers = {
    'aluminum': 'ALI=F', 'copper': 'HG=F', 'zinc': 'ZN=F',
    'tin': 'SN=F', 'lead': 'PB=F', 'steel': 'S00=F',
    'oil': 'BZ=F', 'gas': 'NG=F', 'dxy': 'DX-Y.NYB',
    'usdcny': 'USDCNY=X', 'eurusd': 'EURUSD=X', 'brlusd': 'BRL=X',
    'msci_em': 'EEM', 'xlb': 'XLB', 'pick': 'PICK', 'fxi': 'FXI',
    'alcoa': 'AA', 'century_aluminum': 'CENX', 'hindalco': 'HINDALCO.NS'
}
# Baixa os preços de fecho
market_data = yf.download(list(tickers.values()), start=start_date, end=end_date)['Close']
market_data.columns = list(tickers.keys())
market_data = market_data.reset_index().rename(columns={'Date': 'date'})
print("Dados de mercado do Yahoo Finance recolhidos.")

# --- ETAPA 2: BAIXAR DADOS MACROECONÓMICOS DO FRED ---
fred_series = {
    'us_cpi': 'CPIAUCSL',
    'us_ind_production': 'INDPRO',
    'china_pmi': 'CHNPMI',
    # Adicione outros códigos do FRED aqui se desejar
}
fred_data = pd.DataFrame()
for name, code in fred_series.items():
    try:
        serie = web.DataReader(code, 'fred', start_date, end_date)
        serie.columns = [name]
        if fred_data.empty:
            fred_data = serie
        else:
            fred_data = fred_data.join(serie, how='outer')
    except Exception as e:
        print(f"AVISO: Falhou a recolha de '{name}' do FRED: {e}")
if not fred_data.empty:
    fred_data = fred_data.reset_index().rename(columns={"DATE": "date"})
    print("Dados macroeconómicos do FRED recolhidos.")
    # --- ETAPA 3: UNIR DADOS BRUTOS ---
    df_raw = pd.merge(market_data, fred_data, on="date", how="left")
else:
    df_raw = market_data

# Remove colunas que possam ter falhado o download e ficado totalmente nulas
df_raw.dropna(axis=1, how='all', inplace=True)

# ======================== DIAGNÓSTICO DE VITALIDADE DAS SÉRIES (antes do fill/interp) ========================\n
print("\n### Diagnóstico: Última data COM VALOR REAL (não-NaN) para cada coluna bruta ###")
for col in df_raw.columns:
    if col != 'date':
        dt_ult = df_raw.loc[df_raw[col].notna(), 'date'].max()
        qtd_nulos_final = df_raw[col].tail(30).isna().sum()  # NaNs nas últimas 30 datas
        print(f"{col:25s}  |  Último dado real: {dt_ult}  |  NaNs últimas 30 datas: {qtd_nulos_final}")

# =====================================================================================

# Preenche os valores nulos (ex: fins de semana) usando interpolação linear
df_raw.interpolate(method='linear', limit_direction='both', inplace=True)
df_raw = df_raw.sort_values('date').reset_index(drop=True)

# Padroniza as colunas de metais para USD/tonelada para consistência
if 'aluminum' in df_raw.columns: df_raw['aluminum_usdton'] = df_raw['aluminum'] * 100
if 'copper' in df_raw.columns: df_raw['copper_usdton'] = (df_raw['copper'] / 100) * 2204.62
if 'zinc' in df_raw.columns: df_raw['zinc_usdton'] = df_raw['zinc'] * 100
if 'tin' in df_raw.columns: df_raw['tin_usdton'] = df_raw['tin'] * 100
if 'lead' in df_raw.columns: df_raw['lead_usdton'] = df_raw['lead'] * 100

print("\nPipeline de Coleta de Dados Brutos concluído!")
print(f"DataFrame bruto final com {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas.")
print("Últimos 5 registos de dados:")
display(df_raw.tail())

A recolher dados de 2020-09-16 até 2025-09-16...


HTTP Error 404: 
[**********************79%*************          ]  15 of 19 completedHTTP Error 404: 
[**********************89%******************     ]  17 of 19 completedFailed to get ticker 'S00=F' reason: Failed to perform, curl: (28) Operation timed out after 10005 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[*********************100%***********************]  19 of 19 completed

3 Failed downloads:
['SN=F', 'PB=F', 'S00=F']: YFTzMissingError('possibly delisted; no timezone found')


Dados de mercado do Yahoo Finance recolhidos.
AVISO: Falhou a recolha de 'china_pmi' do FRED: Unable to read URL: https://fred.stlouisfed.org/graph/fredgraph.csv?id=CHNPMI
Response Text:
b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n    <meta charset="utf-8">\r\n    <meta http-equiv="X-UA-Compatible" content="IE=edge">\r\n    <meta name="viewport" content="width=device-width, initial-scale=1">\r\n    <title>Error - St. Louis Fed</title>\r\n    <meta name="description" content="">\r\n    <meta name="keywords" content="">    \r\n    <link rel="stylesheet" type="text/css" href="/assets/bootstrap/dist/css/bootstrap.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/css/home.min.css?1553087253">\r\n    <link rel="stylesheet" type="text/css" href="/assets/fontawesome-free/css/all.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/assets/select2/dist/css/select2.min.css">\r\n    <style>p {\r\n        margin-bottom: 1.5em;\r\n    }</style>\r\n</head>\r\n<body>\r\n<

,date,aluminum,copper,zinc,tin,lead,steel,oil,gas,dxy,...,alcoa,century_aluminum,hindalco,us_cpi,us_ind_production,aluminum_usdton,copper_usdton,zinc_usdton,tin_usdton,lead_usdton
1300,2025-09-09,31.270000,2516.00,5.4203,66.389999,21.740000,97.790001,51.189999,1.176914,39.970001,...,7.1293,90.540001,113.312500,323.364,103.9867,3127.000046,55468.23920,542.030001,6638.999939,2173.999977
1301,2025-09-10,30.930000,2518.25,5.4336,67.489998,22.370001,97.779999,51.540001,1.170275,39.720001,...,7.1209,90.709999,113.515625,323.364,103.9867,3093.000031,55517.84315,543.359995,6748.999786,2237.000084
1302,2025-09-11,32.290001,2587.75,5.4020,66.370003,24.520000,97.540001,52.290001,1.170344,40.740002,...,7.1207,92.610001,113.593750,323.364,103.9867,3229.000092,57050.05405,540.199995,6637.000275,2452.000046
1303,2025-09-12,33.240002,2583.75,5.3882,66.989998,26.090000,97.550003,52.259998,1.173475,40.490002,...,7.1184,91.680000,113.250000,323.364,103.9867,3324.000168,56961.86925,538.819981,6698.999786,2609.000015
1304,2025-09-15,33.599998,2591.00,5.3510,67.440002,26.030001,97.300003,52.639999,1.172649,40.779999,...,7.1242,90.940002,113.406250,323.364,103.9867,3359.999847,57121.70420,535.099983,6744.000244,2603.000069


In [3]:
import numpy as np
# --- Célula 2: Engenharia de Features Completa ---

if 'df_raw' in locals():
    df = df_raw.copy()

    # --- ETAPA 1: DEFINIÇÃO DE VARIÁVEIS E FUNÇÕES ---
    
    # Lista de todos os ativos para criar features
    ativos = [col for col in df.columns if col not in ['date']]
    
    # Função para criar os alvos (targets) dinâmicos
    def add_dynamic_targets(df_in, price_column="aluminum_usdton", horizon=1):
        df_in[f'future_{horizon}d'] = df_in[price_column].shift(-horizon)
        df_in[f'ret_{horizon}d'] = (df_in[f'future_{horizon}d'] - df_in[price_column]) / df_in[price_column]
        
        def classify_bin(x):
            if x > 0.10: return "Sobe muito"
            elif x > 0.03: return "Sobe"
            elif x >= -0.03: return "Na mesma"
            elif x > -0.10: return "Cai"
            else: return "Cai muito"
        
        df_in[f'bin_{horizon}d'] = df_in[f'ret_{horizon}d'].apply(classify_bin)
        return df_in

    # --- ETAPA 2: CRIAÇÃO DAS FEATURES ---

    # 2.1 - Alvos dinâmicos
    horizons = [1, 5, 30, 90]
    for h in horizons:
        df = add_dynamic_targets(df, price_column="aluminum_usdton", horizon=h)

    # 2.2 - Retornos e Log-Retornos
    for ativo in ativos:
        df[f'{ativo}_ret1'] = df[ativo].pct_change()
        df[f'{ativo}_logret1'] = np.log(df[ativo] / df[ativo].shift(1))

    # 2.3 - Rolling Windows e Lags
    rolling_windows = [5, 10, 21]
    lags = [1, 2, 3, 5, 10, 21]
    for ativo in ativos:
        for win in rolling_windows:
            df[f'{ativo}_ma{win}'] = df[ativo].rolling(window=win).mean()
            df[f'{ativo}_std{win}'] = df[ativo].rolling(window=win).std()
            df[f'{ativo}_max{win}'] = df[ativo].rolling(window=win).max()
            df[f'{ativo}_min{win}'] = df[ativo].rolling(window=win).min()
        for lag in lags:
            df[f'{ativo}_lag{lag}'] = df[ativo].shift(lag)

    # 2.4 - Spreads e Ratios (relações entre mercados)
    df['copper_minus_aluminum'] = df.get('copper_usdton', pd.Series(0)) - df.get('aluminum_usdton', pd.Series(0))
    df['oil_minus_gas'] = df.get('oil', pd.Series(0)) - df.get('gas', pd.Series(0))
    df['copper_div_aluminum'] = df.get('copper_usdton', pd.Series(0)) / df.get('aluminum_usdton', pd.Series(0))

    # 2.5 - Indicadores Técnicos específicos do Alumínio
    for win in rolling_windows:
        df[f'alum_zscore_ma{win}'] = (df['aluminum_usdton'] - df[f'aluminum_usdton_ma{win}']) / df[f'aluminum_usdton_std{win}']
        df[f'alum_above_ma{win}'] = (df['aluminum_usdton'] > df[f'aluminum_usdton_ma{win}']).astype(int)
    
    df['alum_cross_ma5_ma21'] = (df['aluminum_usdton_ma5'] > df['aluminum_usdton_ma21']).astype(int)
    df['alum_oversold'] = (df['alum_zscore_ma21'] < -1.5).astype(int)
    df['alum_overbought'] = (df['alum_zscore_ma21'] > 1.5).astype(int)

    # 2.6 - Features de Calendário
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['weekday'] = df['date'].dt.weekday
    df['end_of_month'] = df['date'].dt.is_month_end.astype(int)
    
    # --- ETAPA 3: LIMPEZA FINAL ---
    
    # Remove colunas duplicadas que podem ter sido criadas
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Substitui valores infinitos (de divisões por zero) por NaN e depois preenche
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(method='ffill', inplace=True)
    df.dropna(inplace=True) # Remove as primeiras linhas que ainda têm NaNs dos lags/rolling
    df = df.reset_index(drop=True)

    # Salva o DataFrame final
    output_filename = 'aluminium_full_featured_retrained.csv'
    df.to_csv(output_filename, index=False)

    print(f"\nPipeline de Engenharia de Features concluído!")
    print(f"DataFrame final com {df.shape[0]} linhas e {df.shape[1]} colunas.")
    print(f"Ficheiro salvo como: '{output_filename}'")
    display(df.head())

else:
    print("ERRO: DataFrame 'df_raw' não foi criado. Execute a Célula 1 primeiro.")


Pipeline de Engenharia de Features concluído!
DataFrame final com 1284 linhas e 512 colunas.
Ficheiro salvo como: 'aluminium_full_featured_retrained.csv'


,date,aluminum,copper,zinc,tin,lead,steel,oil,gas,dxy,...,alum_above_ma10,alum_zscore_ma21,alum_above_ma21,alum_cross_ma5_ma21,alum_oversold,alum_overbought,month,quarter,weekday,end_of_month
0,2020-10-15,11.780352,1861.50,5.5905,43.160000,7.83,93.860001,40.557194,1.174398,38.627689,...,0,0.134996,1,1,0,0,10,4,3,0
1,2020-10-16,12.086832,1871.75,5.6112,42.930000,7.66,93.680000,40.700123,1.170713,39.214436,...,1,0.827811,1,1,0,0,10,4,4,0
2,2020-10-19,12.096408,1858.00,5.6448,42.619999,7.50,93.430000,40.583992,1.171550,39.196659,...,1,1.014807,1,1,0,0,10,4,0,0
3,2020-10-20,12.259228,1841.00,5.6056,43.160000,7.73,93.070000,41.030651,1.176886,39.365566,...,1,1.268684,1,1,0,0,10,4,1,0
4,2020-10-21,12.757259,1851.50,5.6042,41.730000,7.61,92.610001,41.146782,1.182984,39.721172,...,1,1.972979,1,1,0,1,10,4,2,0
